# 第 7 章：Mini GPT 从零实现

这个 notebook 对应 `lessons/07_mini_gpt.md`，把 tokenizer、embedding、position embedding、Transformer block、LM head、generate 和 checkpoint 串成一个最小 GPT。

In [ ]:
import tempfile
from pathlib import Path

import torch

from src.models.mini_gpt import (
    MiniGPT,
    MiniGPTConfig,
    MiniGPTTrainingConfig,
    generate,
    load_checkpoint,
    save_checkpoint,
    train_mini_gpt,
)
from src.tokenizer.simple_tokenizer import CharacterTokenizer

## 1. Tokenizer 与 Model Config

Mini GPT 的 config 必须保存 vocab size、context length、hidden dim、层数、head 数和 dropout。

In [ ]:
text = "你好 GPT。你好 LLM。"
tokenizer = CharacterTokenizer.from_texts([text])
config = MiniGPTConfig(
    vocab_size=tokenizer.vocab_size,
    block_size=8,
    hidden_dim=16,
    num_layers=1,
    num_heads=4,
)
model = MiniGPT(config)
print(config)
print("vocab size:", tokenizer.vocab_size)

## 2. Forward 与 Loss

输入 shape 是 `(B, T)`，logits shape 是 `(B, T, V)`。

In [ ]:
token_ids = torch.tensor(tokenizer.encode(text), dtype=torch.long)
input_ids = token_ids[:8].unsqueeze(0)
labels = token_ids[1:9].unsqueeze(0)
logits, loss = model(input_ids, labels)
print("logits:", logits.shape)
print("loss:", round(loss.item(), 4))

## 3. Causal Leakage 检查

改变未来 token 不应影响更早位置的 logits。

In [ ]:
model.eval()
original = input_ids.clone()
changed = original.clone()
changed[0, -2:] = torch.tensor([tokenizer.eos_token_id, tokenizer.eos_token_id])
logits_a, _ = model(original)
logits_b, _ = model(changed)
print(torch.allclose(logits_a[:, :-2], logits_b[:, :-2], atol=1e-6))

## 4. Tiny Corpus Training

小语料 loss 下降证明整条 GPT 管线能训练。

In [ ]:
train_config = MiniGPTTrainingConfig(seed=0, batch_size=8, lr=0.01, steps=40)
trained_model, history = train_mini_gpt(token_ids.repeat(16), config, train_config)
print("first loss:", round(history.losses[0], 4))
print("last loss:", round(history.losses[-1], 4))

## 5. Generate

生成时只取最后一个位置的 logits，并把上下文裁剪到 `block_size`。

In [ ]:
prompt = torch.tensor(tokenizer.encode("你好", add_special_tokens=True), dtype=torch.long)
generated = generate(
    trained_model,
    prompt,
    max_new_tokens=8,
    top_k=3,
    generator=torch.Generator().manual_seed(0),
)
print(generated.tolist())
print(tokenizer.decode(generated))

## 6. Checkpoint Round Trip

Checkpoint 必须同时保存模型 config、权重和 tokenizer。

In [ ]:
with tempfile.TemporaryDirectory() as tmpdir:
    path = Path(tmpdir) / "mini_gpt.pt"
    save_checkpoint(path, trained_model, tokenizer, extra={"step": history.items[-1].step})
    loaded_model, loaded_tokenizer, extra = load_checkpoint(path)

print(loaded_model.config)
print(loaded_tokenizer.token_to_id == tokenizer.token_to_id)
print(extra)